<a id="cellular-automaton"></a>

$$
\text{元胞自动机 (Cellular Automaton, CA): 时间, 空间, 状态都离散的动力学模型} \\ \quad \\
\text{四个要素:} \\
\text{1. 元胞 (Cell) 和网格 (Lattice): 空间被分成一个个格子, 例如一维的一排, 二维的棋盘} \\
\text{2. 状态 (State): 每个元胞在每个时刻处于有限个状态之一, 例如 0 (死) / 1 (活)} \\
\text{3. 邻域 (Neighborhood): 决定一个元胞下一步状态的周围元胞} \\
\quad \text{冯诺依曼邻域 (von Neumann): 上下左右 4 个; 摩尔邻域 (Moore): 周围 8 个} \\
\text{4. 局部规则 (Local Rule): } s_{t+1}(i) = f\big(s_t(i), s_t(\text{邻居})\big) \text{, 所有元胞同时 (同步 Synchronous) 更新} \\ \quad \\
\text{边界条件: 固定边界 (边界外视为 0), 或周期边界 (Periodic, 左右/上下首尾相接, 网格变成一个环面 Torus)} \\ \quad \\
\text{和微分方程的对比: ODE / PDE 描述 "连续量的变化率", 元胞自动机直接给出 "离散状态的更新规则"} \\
\text{适合: 交通流, 森林火灾, 传染病的空间扩散, 晶体生长, 城市扩张等 "局部相互作用产生整体模式" 的问题}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d
from scipy.special import comb

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

von_neumann = [(-1, 0), (1, 0), (0, -1), (0, 1)]
moore = [(di, dj) for di in (-1, 0, 1) for dj in (-1, 0, 1) if (di, dj) != (0, 0)]
print(f'冯诺依曼邻域 {len(von_neumann)} 个: {von_neumann}')
print(f'摩尔邻域 {len(moore)} 个: {moore}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, offsets, title in [(axs[0], von_neumann, 'von Neumann'), (axs[1], moore, 'Moore')]:
    grid = np.zeros((5, 5))
    grid[2, 2] = 2
    for di, dj in offsets:
        grid[2 + di, 2 + dj] = 1
    ax.imshow(grid, cmap='Blues')
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

<a id="elementary-ca"></a>

$$
\text{一维初等元胞自动机 (Elementary Cellular Automaton)} \\
\text{状态 \{0, 1\}, 邻域 = (左, 自己, 右), 共 } 2^3 = 8 \text{ 种组合, 每种组合规定一个新状态 (0 或 1)} \\
\text{所以一共有 } 2^8 = 256 \text{ 种规则} \\ \quad \\
\text{规则编号 (Wolfram Code): 把组合 (l, c, r) 看成二进制数 } k = 4l + 2c + r \in \{0, \cdots, 7\} \\
\text{规则编号 N 的二进制第 k 位就是组合 k 的新状态: } s_{t+1}(i) = \left\lfloor \frac{N}{2^k} \right\rfloor \bmod 2 \\
\text{例: Rule 90 = } 01011010_2 \text{, 组合 111, 110, 101, 100, 011, 010, 001, 000 依次得到 0, 1, 0, 1, 1, 0, 1, 0} \\
\text{也就是 } s_{t+1}(i) = s_t(i - 1) \oplus s_t(i + 1) \text{ (左右邻居异或 XOR)} \\ \quad \\
\text{Rule 90 从单个 1 开始: 第 t 行第 } i \text{ 个位置的值 } = \binom{t}{(t + i)/2} \bmod 2 \text{ (帕斯卡三角形模 2, 谢尔宾斯基三角形)} \\
\text{推导: XOR 就是模 2 加法, 状态更新 } s_{t+1}(i) = s_t(i-1) + s_t(i+1) \pmod 2 \text{ 和杨辉三角的递推 } \binom{t+1}{k} = \binom{t}{k-1} + \binom{t}{k} \text{ 一样} \\ \quad \\
\text{Rule 30: 看起来完全随机 (曾被用作随机数生成器); Rule 110: 被证明是图灵完备的 (可以模拟任何计算)}
$$

数学基础: [二项分布与组合数](../math_basics/statistics.ipynb#binomial)

In [ ]:
def elementary_ca(rule, width=161, steps=80, init=None):
    table = np.array([(rule >> k) & 1 for k in range(8)])     # table[k] = 组合 k 的新状态
    row = np.zeros(width, dtype=int)
    if init is None:
        row[width // 2] = 1
    else:
        row = init.copy()
    history = [row]
    for _ in range(steps):
        left, right = np.roll(row, 1), np.roll(row, -1)          # 周期边界
        row = table[4 * left + 2 * row + right]
        history.append(row)
    return np.array(history)

print(f'Rule 90 的查找表 (组合 0..7) = {[(90 >> k) & 1 for k in range(8)]}')
ca90 = elementary_ca(90)
# 独立验证 1: Rule 90 == 左右邻居异或
row = ca90[10]
print(f'Rule 90 == 左右异或 ? {np.array_equal(ca90[11], np.roll(row, 1) ^ np.roll(row, -1))}')
# 独立验证 2: 帕斯卡三角形模 2
width, center = 161, 80
pascal = np.zeros_like(ca90)
for t in range(ca90.shape[0]):
    for i in range(width):
        offset = i - center
        if abs(offset) <= t and (t + offset) % 2 == 0:
            pascal[t, i] = int(comb(t, (t + offset) // 2, exact=True) % 2)
print(f'Rule 90 == C(t, (t+i)/2) mod 2 ? {np.array_equal(ca90, pascal)}')

fig, axs = plt.subplots(1, 3, figsize=(12, 3))
for ax, rule in zip(axs, [30, 90, 110]):
    ax.imshow(elementary_ca(rule), cmap='binary', interpolation='nearest')
    ax.set_title(f'Rule {rule}')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

# Rule 30 的中间一列作为随机比特
center_bits = elementary_ca(30, width=2001, steps=1000)[:, 1000]
print(f'Rule 30 中间列 1000 个比特中 1 的比例 = {center_bits.mean():.3f}')

<a id="game-of-life"></a>

$$
\text{康威生命游戏 (Conway's Game of Life): 二维, 状态 \{0 = 死, 1 = 活\}, 摩尔邻域 (8 个邻居)} \\
\text{设 } N_t(i, j) = \sum_{(di, dj) \in \text{Moore}} s_t(i + di, j + dj) \text{ 为活邻居数, 规则 B3/S23:} \\
\text{B3 (Birth): 死细胞恰好有 3 个活邻居 → 变活 (繁殖)} \\
\text{S23 (Survival): 活细胞有 2 或 3 个活邻居 → 存活; 少于 2 个 → 死亡 (孤独), 多于 3 个 → 死亡 (拥挤)} \\
s_{t+1} = \begin{cases} 1, & N_t = 3 \\ s_t, & N_t = 2 \\ 0, & \text{其他} \end{cases} \\ \quad \\
\text{计算邻居数的两种方法:} \\
\text{1. 把整个网格平移 8 次再相加 (np.roll 自带周期边界)} \\
\text{2. 卷积 (Convolution): 用卷积核 } K = \begin{bmatrix} 1 & 1 & 1 \\ 1 & 0 & 1 \\ 1 & 1 & 1 \end{bmatrix} \text{ 和网格做二维卷积, } N = s * K \\
\quad (\text{卷积的每个输出就是 "核覆盖的区域内加权求和", 中心权重为 0 正好不算自己; CNN 的卷积层是同一个运算})
$$

In [ ]:
def life_step_loop(grid):
    n, m = grid.shape
    new = np.zeros_like(grid)
    for i in range(n):
        for j in range(m):
            count = sum(grid[(i + di) % n, (j + dj) % m] for di, dj in moore)
            if count == 3 or (count == 2 and grid[i, j] == 1):
                new[i, j] = 1
    return new

def life_step_roll(grid):
    count = sum(np.roll(np.roll(grid, di, axis=0), dj, axis=1) for di, dj in moore)
    return ((count == 3) | ((count == 2) & (grid == 1))).astype(int)

KERNEL = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]])
def life_step_conv(grid):
    count = convolve2d(grid, KERNEL, mode='same', boundary='wrap')
    return ((count == 3) | ((count == 2) & (grid == 1))).astype(int)

rng = np.random.default_rng(0)
g_loop = g_roll = g_conv = (rng.random((30, 30)) < 0.35).astype(int)
same = True
for _ in range(50):
    g_loop, g_roll, g_conv = life_step_loop(g_loop), life_step_roll(g_roll), life_step_conv(g_conv)
    same &= np.array_equal(g_loop, g_roll) and np.array_equal(g_roll, g_conv)
print(f'50 步内, 双重循环 == np.roll == convolve2d ? {same}')

<a id="life-patterns"></a>

$$
\text{典型图案 (Patterns)} \\
\text{1. 静物 (Still Life): 周期 1, 例如方块 (Block, 2x2): 每个活细胞恰好 3 个活邻居, 每个周围的死细胞最多 2 个} \\
\text{2. 振荡器 (Oscillator): 周期 } p > 1 \text{, 例如闪光灯 (Blinker, 一行 3 个): 横 → 竖 → 横, 周期 2} \\
\text{3. 滑翔机 (Glider): 每 4 代恢复原来的形状, 并且整体沿对角线移动 (1, 1) 格, 速度 c/4} \\ \quad \\
\text{判断周期: 找最小的 } p \text{ 和平移量 } (\Delta i, \Delta j) \text{, 使得 } s_{t+p}(i, j) = s_t(i - \Delta i, j - \Delta j) \text{ 对所有格子成立}
$$

In [ ]:
def place(pattern, size=12, at=(3, 3)):
    grid = np.zeros((size, size), dtype=int)
    p = np.array(pattern)
    grid[at[0]:at[0] + p.shape[0], at[1]:at[1] + p.shape[1]] = p
    return grid

def find_period(grid, max_period=10):
    n, m = grid.shape
    g = grid.copy()
    for p in range(1, max_period + 1):
        g = life_step_conv(g)
        for di in range(-3, 4):
            for dj in range(-3, 4):
                if np.array_equal(g, np.roll(np.roll(grid, di, axis=0), dj, axis=1)):
                    return p, (di, dj)
    return None, None

block = place([[1, 1], [1, 1]])
blinker = place([[1, 1, 1]])
glider = place([[0, 1, 0], [0, 0, 1], [1, 1, 1]])
for name, g in [('方块 block', block), ('闪光灯 blinker', blinker), ('滑翔机 glider', glider)]:
    period, shift = find_period(g)
    print(f'{name}: 周期 = {period}, 每个周期的平移 = {shift}')

fig, axs = plt.subplots(1, 5, figsize=(12, 3))
g = glider.copy()
for k, ax in enumerate(axs):
    ax.imshow(g, cmap='binary')
    ax.set_title(f'glider t = {k}')
    ax.set_xticks([]); ax.set_yticks([])
    g = life_step_conv(g)
plt.tight_layout()
plt.show()

In [ ]:
# 随机初始状态的演化: 活细胞比例下降, 最后剩下静物和振荡器
rng = np.random.default_rng(1)
grid = (rng.random((100, 100)) < 0.3).astype(int)
snapshots, density = {}, []
for t in range(301):
    if t in (0, 20, 300):
        snapshots[t] = grid.copy()
    density.append(grid.mean())
    grid = life_step_conv(grid)

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
for ax, (t, snap) in zip(axs[:3], snapshots.items()):
    ax.imshow(snap, cmap='binary')
    ax.set_title(f't = {t}')
    ax.set_xticks([]); ax.set_yticks([])
axs[3].plot(density, label='live fraction')
axs[3].set_xlabel('generation')
plot_nicely()
plt.show()
print(f'初始活细胞比例 = {density[0]:.3f}, 300 代后 = {density[-1]:.3f}')

<a id="forest-fire"></a>

$$
\text{建模应用: 森林火灾与渗流 (Percolation)} \\
\text{状态: 0 = 空地, 1 = 树, 2 = 燃烧, 3 = 烧毁; 冯诺依曼邻域} \\
\text{规则: 燃烧 → 烧毁; 树的 4 个邻居中有燃烧的 → 燃烧; 其他不变} \\
\text{初始: 每个格子以概率 p (森林密度) 是树, 点燃最左边一列的树} \\ \quad \\
\text{问题: 森林密度多大时, 火能从左边一直烧到右边?} \\
\text{火只能沿相邻的树传播, 所以最终烧毁的区域 = 和最左列连通的树的连通分量 (Connected Component)} \\
\text{这是二维正方形网格上的格点渗流 (Site Percolation), 存在临界密度 } p_c \approx 0.5927 \text{:} \\
p < p_c \text{ 时火很快熄灭, } p > p_c \text{ 时火几乎总能烧穿整片森林, 在 } p_c \text{ 附近发生 "相变" (Phase Transition)} \\
\text{(同样的模型可以描述: 传染病在人群中的扩散, 多孔介质中的液体渗透, 谣言传播)}
$$

In [ ]:
from scipy.ndimage import label

def burn(forest):
    state = forest.copy()                   # 0 空地, 1 树
    state[:, 0][state[:, 0] == 1] = 2       # 点燃最左列
    steps = 0
    while np.any(state == 2):
        burning = state == 2
        neighbor_burning = np.zeros_like(burning)
        neighbor_burning[1:, :] |= burning[:-1, :]
        neighbor_burning[:-1, :] |= burning[1:, :]
        neighbor_burning[:, 1:] |= burning[:, :-1]
        neighbor_burning[:, :-1] |= burning[:, 1:]
        state[burning] = 3
        state[(state == 1) & neighbor_burning] = 2
        steps += 1
    return state, steps

def burned_by_components(forest):
    # 独立方法: 4-连通分量中, 包含最左列树木的那些分量
    labels, _ = label(forest)               # 默认结构元素就是 4-连通 (冯诺依曼)
    left_labels = np.unique(labels[:, 0][labels[:, 0] > 0])
    return np.isin(labels, left_labels)

rng = np.random.default_rng(2)
forest = (rng.random((80, 80)) < 0.6).astype(int)
final, steps = burn(forest)
print(f'元胞自动机烧毁区域 == 连通分量 ? {np.array_equal(final == 3, burned_by_components(forest))}, 燃烧了 {steps} 步')

densities = np.linspace(0.4, 0.8, 21)
cross_prob, burned_frac = [], []
for p in densities:
    crosses, fracs = 0, []
    for _ in range(40):
        f = (rng.random((80, 80)) < p).astype(int)
        burned = burned_by_components(f)
        crosses += burned[:, -1].any()
        fracs.append(burned.sum() / max(f.sum(), 1))
    cross_prob.append(crosses / 40)
    burned_frac.append(np.mean(fracs))

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
show = np.where(final == 3, 2, forest)      # 0 空地, 1 未烧的树, 2 烧毁
axs[0].imshow(show, cmap='YlOrBr', interpolation='nearest')
axs[0].set_title('p = 0.6, burned area')
axs[0].set_xticks([]); axs[0].set_yticks([])
axs[1].plot(densities, cross_prob, 'o-', label='P(fire reaches right edge)')
axs[1].plot(densities, burned_frac, 's-', label='fraction of trees burned')
axs[1].axvline(0.5927, color='r', linestyle='--', label='p_c = 0.5927')
axs[1].set_xlabel('forest density p')
plot_nicely()
plt.show()
print(f'密度 0.50: 烧穿概率 = {cross_prob[5]:.2f}; 密度 0.70: 烧穿概率 = {cross_prob[15]:.2f}')